In [ ]:
import pandas as pd

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import psycopg2
import re



#analytics_mongo = client_db.get_database('analytics_db')

#db = client.get_database('telematics_db')

#db2 = client2.get_database('telematics_db')

#wave=app_mongo.get_database('app_db')



In [ ]:
vehicle_numbers=pd.read_sql("select vehicle_number from fleet_lake.vehicles",con)['vehicle_number'].tolist()
vehicle_numbers=[i for i in vehicle_numbers if i!=None]
vehicle_numbers=[re.sub("'", '', i) for i in vehicle_numbers if i!=None]


In [ ]:
len(vehicle_numbers)

In [ ]:
make_model=pd.DataFrame()
for i in range(0,len(vehicle_numbers)+10000,10000):
    reuse=pd.read_sql("select registration_no,maker_model from fleet_lake.vahaan where registration_no in {}".format(tuple(vehicle_numbers[i:i+10000])),con)
    make_model=pd.concat([make_model,reuse])
    print(i)
    

In [ ]:
make_model.dropna(inplace=True)

In [ ]:
make_model.drop_duplicates('registration_no',inplace=True)

In [ ]:
make_model.rename(columns={'registration_no':'vehicle_number','maker_model':'make/model'},inplace=True)

In [ ]:
#make_model=pd.DataFrame(total_list,columns=['vehicle_number','make/model'])
make_model=make_model[make_model['make/model'].notnull()]
make_model=make_model[make_model['make/model']!=" "]
make_model=make_model[make_model['make/model']!=""]

make_model.drop_duplicates('vehicle_number',inplace=True)

In [ ]:
make_model

In [ ]:


#SANATIZING
make_model['type']=make_model['make/model'].apply(lambda x: str(type(x)))
make_model=make_model[make_model['type']!="<class 'datetime.datetime'>"]
del make_model['type']

make_model['contains_upto']=make_model['make/model'].str.findall('upto').apply(len)
make_model=make_model[make_model['contains_upto']==0]
del make_model['contains_upto']

In [ ]:
#SPLITING MAKE
make_model['make']=make_model['make/model'].apply(lambda x : x.split('/')[0])


In [ ]:
make_model['contains_numbers']=make_model['make'].str.findall(r'\d\d').apply(len)
make_model=make_model[make_model['contains_numbers']==0]
del make_model['contains_numbers']

In [ ]:
def split_data(x):
    try:
        a=x.split('/')[1:]
        b=''
        for i in a:
            b=b+i
        return b
    except:
        a=None
        return a

In [ ]:
make_model['model']=make_model['make/model'].apply(split_data)


In [ ]:
make_model=make_model[make_model['model'].notnull()]

In [ ]:
grid=pd.read_csv("Grid - Sheet1.csv")

In [ ]:
grid=grid[~grid['correction'].isin(['LPK','LPT'])]

In [ ]:
grid[-1:]

In [ ]:
total=''
for i in grid[grid['type'].isin(['4 digs'])]['correction'].unique():
    a=i[0:2]+'\D'+i[2:4]
    total=total+"|"+a
    total=total+"|"+i


In [ ]:
for i in grid[grid['type'].isin(['3 digs','3 dig','5 dig'])]['correction'].unique():
    total=total+"|"+i


In [ ]:
for i in grid[grid['type'].isin(['text'])]['correction'].unique():
    total=total+"|"+"(?i)"+i


In [ ]:
exclusive_models=['torro25','traco35','truxo25','truxo31','torro.25','traco.35','truxo.25','truxo.31','Blazo37','Blazo25','Blazo40','Blazo49','Blazo.37','Blazo.25','Blazo.40','Blazo.49','load.king','loadking','pickup','pick.up']    



In [ ]:
for i in exclusive_models:
    total=total+"|"+"(?i)"+i

    

In [ ]:
make_model['model_regex']=make_model['model'].str.findall('r'+str(total[1:]))


In [ ]:
make_model['model_regex']=make_model['model_regex'].apply(lambda x: x if len(x)>0 else None)

In [ ]:
import re
def model_create(model):
    if model!=None:
        final=''
        for i in model: 
            i = re.sub('\.', '', i)
            i= re.sub('-', '', i)
            i= re.sub(' ', '', i)
            i= re.sub(':', '', i)
            i= re.sub("\*", '', i)
            i=i.upper()
            if i not in final.split(' '):
                final=final+" "+i
        alphabatecally=''
        for i in reversed(sorted(final.split(' '))):
            alphabatecally=alphabatecally+" "+i
            
        return alphabatecally[1:]
    else:
        return None

        
    

In [ ]:
make_model['final_model']=make_model['model_regex'].apply(model_create)

In [ ]:
make_model.final_model.isna().value_counts()

In [ ]:
make_model.head()

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
def OEM_categorisation(x):
    name_list=[ 'ASHOK LEYLAND',
        'TATA',
 'VE COMMERCIAL',
 'MAHINDRA',
 'EICHER',
 'DAIMLER',
 'SML ISUZU',
 'VECTRA',
 'Baldev',
 'FORCE',
 'PIAGGIO',
 'MAN TRUCKS',
 'AMW',
 'GENERAL']
    for i in name_list:
        if fuzz.partial_ratio(x,i)>70:
            #print('compared ' + i + ' with '+ x +' and SELECTED')
            return i 
        else:
            None
            #print('compared ' + i + ' with '+ x +' and rejected')
    return x
                
    
    

In [ ]:
make_model=make_model[~make_model['make'].isin([' ','  ','   ','    ','     '])]

In [ ]:
make_model['sanitized_make']=make_model.make.apply(OEM_categorisation)

In [ ]:
selected_make_model=make_model[(make_model['final_model'].notnull())]

In [ ]:
selected_make_model.sanitized_make.value_counts()

In [ ]:
selected_make_model['final_make_model']=selected_make_model['sanitized_make']+' '+selected_make_model['final_model']

In [ ]:
selected_make_model

In [ ]:
engine.execute("drop table vehicle_make_model ")

In [ ]:
selected_make_model.set_index(['vehicle_number']).to_sql("vehicle_make_model",con=engine,if_exists='append',chunksize=1000,method='multi')

In [ ]:
selected_make_model

In [ ]:
list_of_vehicles_in_utilisation_data=list(pd.read_sql("select vehicle_number from utilisation_data where trailing_quater=1",engine)['vehicle_number'])
list_of_vehicles_in_utilisation_data=[i for i in list_of_vehicles_in_utilisation_data if i!=None]
utilisation_data=pd.DataFrame()
for i in range (0,len(list_of_vehicles_in_utilisation_data)+1000,1000):
    a=pd.read_sql("select * from utilisation_data where vehicle_number in {}".format(tuple(list_of_vehicles_in_utilisation_data[i:i+1000])),engine)
    utilisation_data=pd.concat([a,utilisation_data])
    print(i)

In [ ]:
utilisation_data.head()
util_backup.head()

In [ ]:
make_model=make_model[['vehicle_number','trial']]


In [ ]:
make_model=make_model.rename(columns={'trial':'corrected_model'})

In [ ]:
del utilisation_data['corrected_model']

In [ ]:
del utilisation_data['make_model_final']

In [ ]:
utilisation_data_merged=pd.merge(utilisation_data,make_model,on='vehicle_number',how='left')

In [ ]:
utilisation_data_merged['make_model_final']=utilisation_data_merged['sanitized_make']+ ' ' +utilisation_data_merged['corrected_model']

In [ ]:
utilisation_data_merged

In [ ]:
engine.execute("delete from  utilisation_data")

In [ ]:
utilisation_data_merged.set_index(['vehicle_number','trailing_quater','vehicle_id']).to_sql("utilisation_data",con=engine,if_exists='append',chunksize=5000,method='multi')  


In [ ]:
pd.read_sql("select count(*) from utilisation_data",engine)

In [ ]:
pd.read_sql("select * from vehicle_make_model limit 100",engine)

# make model from comeplete data in vehicle_make_model analytics db


In [ ]:
GPS_vehicles=pd.read_sql("select vehicle_number from analytics.vehicle_details where installation_date<'2020-01-01'",con)

In [ ]:
make_models=pd.read_sql("select vehicle_number,final_model,final_make_model,sanitized_make from vehicle_make_model where vehicle_number in {}".format(tuple(GPS_vehicles['vehicle_number'])),engine)  



In [ ]:
make_models.vehicle_number.nunique()

In [ ]:
make_models.rename(columns={'final_model':'corrected_model','final_make_model':'make_model_final'},inplace=True)

In [ ]:
make_models['state_of_reg']=make_models['vehicle_number'].apply(lambda x : x[0:2])

In [ ]:
make_models.to_csv("data/make_model_2020_Q1.csv")

In [ ]:
make_models